In [1]:
import os
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your project-root .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Imports and Groq initialization successful.")

Imports and Groq initialization successful.


In [2]:
event_name = "College Cultural Day"
location = "Solapur, Maharashtra, India"
event_type = "Outdoor cultural event"
event_date = "2026-10-24"

print("Event:", event_name)
print("Location:", location)
print("Type:", event_type)
print("Date:", event_date)

Event: College Cultural Day
Location: Solapur, Maharashtra, India
Type: Outdoor cultural event
Date: 2026-10-24


In [3]:
def get_coordinates(location):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    response = requests.get(
        url,
        params={
            "name": location.split(",")[0],
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=20
    )

    response.raise_for_status()
    data = response.json()

    results = data.get("results", [])

    if not results:
        raise ValueError(f"No location found for: {location}")

    # Prefer a result matching the requested country when possible.
    india_results = [
        item for item in results
        if item.get("country_code") == "IN"
    ]

    candidates = india_results or results

    # For this example, choose the result whose name matches the city.
    city_name = location.split(",")[0].strip().lower()
    exact_matches = [
        item for item in candidates
        if item.get("name", "").lower() == city_name
    ]

    place = (exact_matches or candidates)[0]

    return {
        "name": place.get("name"),
        "country": place.get("country"),
        "latitude": place["latitude"],
        "longitude": place["longitude"]
    }


coordinates = get_coordinates(location)
print("Resolved location:", coordinates)

Resolved location: {'name': 'Solapur', 'country': 'India', 'latitude': 17.67152, 'longitude': 75.91044}


In [4]:
def get_weather_forecast(latitude, longitude, event_date):
    url = "https://api.open-meteo.com/v1/forecast"

    response = requests.get(
        url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max,weather_code",
            "timezone": "Asia/Kolkata",
            "start_date": event_date,
            "end_date": event_date
        },
        timeout=20
    )

    response.raise_for_status()
    data = response.json()

    if not data.get("daily") or not data["daily"].get("time"):
        raise ValueError(
            "No daily forecast returned for this date. "
            "The date may be outside the forecast range."
        )

    daily = data["daily"]

    return {
        "date": daily["time"][0],
        "temperature_max_c": daily["temperature_2m_max"][0],
        "temperature_min_c": daily["temperature_2m_min"][0],
        "precipitation_probability_percent": daily[
            "precipitation_probability_max"
        ][0],
        "weather_code": daily["weather_code"][0]
    }


weather_data = get_weather_forecast(
    coordinates["latitude"],
    coordinates["longitude"],
    event_date
)

print("REAL WEATHER FORECAST")
print(weather_data)

REAL WEATHER FORECAST
{'date': '2026-10-24', 'temperature_max_c': 34.3, 'temperature_min_c': 24.8, 'precipitation_probability_percent': 2, 'weather_code': 3}


In [5]:
class EventRecommendation(BaseModel):
    recommendation: str = Field(
        description="Recommended event arrangement based on the forecast"
    )
    weather_risks: list[str] = Field(
        description="Weather-related risks supported by the forecast"
    )
    precautions: list[str] = Field(
        description="Practical precautions for the event"
    )
    forecast_date: str = Field(
        description="The date of the weather forecast used"
    )


weather_parser = PydanticOutputParser(
    pydantic_object=EventRecommendation
)

print("Weather recommendation schema created.")

Weather recommendation schema created.


In [6]:
recommendation_prompt = PromptTemplate(
    template="""
You are an event planning assistant.

Recommend arrangements for the event using the supplied event details
and actual weather forecast.

Rules:
- Base weather-related statements only on the supplied forecast.
- Consider precipitation probability and temperature.
- Do not invent weather conditions.
- If the forecast is incomplete, explain the uncertainty.
- Provide practical precautions appropriate to the event.
- Do not claim that a venue or backup facility has already been booked.
- Return the required structured output.

Event name: {event_name}
Location: {location}
Event type: {event_type}
Event date: {event_date}

Weather forecast:
{weather_data}

{format_instructions}
""",
    input_variables=[
        "event_name",
        "location",
        "event_type",
        "event_date",
        "weather_data"
    ],
    partial_variables={
        "format_instructions": weather_parser.get_format_instructions()
    }
)

recommendation_chain = recommendation_prompt | llm | weather_parser

print("Weather recommendation chain created.")

Weather recommendation chain created.


In [7]:
recommendation = recommendation_chain.invoke({
    "event_name": event_name,
    "location": location,
    "event_type": event_type,
    "event_date": weather_data["date"],
    "weather_data": str(weather_data)
})

print("WEATHER-ADJUSTED EVENT RECOMMENDATION")
print(recommendation.model_dump_json(indent=2))

WEATHER-ADJUSTED EVENT RECOMMENDATION
{
  "recommendation": "Proceed with the College Cultural Day as an outdoor event on 2026-10-24, but incorporate ample shade structures, hydration stations, and sun protection measures to keep participants comfortable in warm conditions.",
  "weather_risks": [
    "High daytime temperature (max 34.3 °C) increases risk of heat‑related discomfort or illness.",
    "Strong sun exposure due to likely clear/partly‑cloudy skies (weather code 3).",
    "Although precipitation probability is low (2 %), a brief shower cannot be completely ruled out."
  ],
  "precautions": [
    "Set up shade canopies, tents, or temporary pavilions over performance and audience areas.",
    "Provide multiple water refill stations and encourage regular fluid intake.",
    "Offer sunscreen dispensers and advise attendees to wear hats or light, breathable clothing.",
    "Station a first‑aid kit and designate staff to monitor signs of heat exhaustion.",
    "Keep a simple rain‑p

In [8]:
assert isinstance(recommendation, EventRecommendation)
assert recommendation.recommendation.strip()
assert recommendation.forecast_date == weather_data["date"]
assert isinstance(recommendation.weather_risks, list)
assert isinstance(recommendation.precautions, list)

print("Validation passed.")
print("A real weather forecast was retrieved.")
print("The recommendation was generated from the forecast and event details.")

Validation passed.
A real weather forecast was retrieved.
The recommendation was generated from the forecast and event details.


In [9]:
print("""
Conclusion:
The Weather-Adjusted Event Recommendation Pipeline retrieves location
coordinates using Open-Meteo geocoding, obtains a real weather forecast,
and passes the event details and forecast to a LangChain prompt.
PydanticOutputParser validates the recommendation structure. The result
supports event planning but should be reviewed against updated forecasts.
""")


Conclusion:
The Weather-Adjusted Event Recommendation Pipeline retrieves location
coordinates using Open-Meteo geocoding, obtains a real weather forecast,
and passes the event details and forecast to a LangChain prompt.
PydanticOutputParser validates the recommendation structure. The result
supports event planning but should be reviewed against updated forecasts.

